# Notebook 08 -- Web App Integration

## Data-Driven Framework for Early Detection of Alzheimer's Disease Using MRI Brain Images

---

### What This Notebook Does
1. Verify deployment files exist (`deployment/best_model.pth`, `deployment/deploy_config.json`)
2. Test the prediction pipeline (dry run)
3. Explain the `app.py` Streamlit web app features
4. Generate `requirements.txt`
5. Instructions to run and deploy

### Web App Features
- **Premium dark theme UI** with gradient styling
- **Patient info fields** (name, ID) for personalized reports
- **Upload MRI brain image** for AI-powered classification
- **Real-time prediction** with confidence scores
- **Dark-themed probability bar chart**
- **Downloadable HTML report** with embedded MRI scan, patient info, and results
- **Model performance tab** with confusion matrix and per-class metrics
- **Sidebar** with model info and disclaimer

### How to Run
```bash
streamlit run app.py
```

## 1. Verify Deployment Files

In [1]:
import os
import json
from pathlib import Path

DEPLOY_DIR = Path("deployment")
PROJECT_DIR = Path(".")

# Verify deployment files exist
model_path = DEPLOY_DIR / 'best_model.pth'
config_path = DEPLOY_DIR / 'deploy_config.json'
eval_path = Path('models') / 'evaluation_results.json'

assert model_path.exists(), f"Model not found at {model_path}. Run Notebook 07 first!"
assert config_path.exists(), f"Config not found at {config_path}. Run Notebook 07 first!"

with open(config_path, 'r') as f:
    deploy_config = json.load(f)

print("Deployment files verified:")
print(f"  Model:   {model_path} ({model_path.stat().st_size / 1024**2:.1f} MB)")
print(f"  Config:  {config_path}")
print(f"  Eval:    {eval_path} (exists: {eval_path.exists()})")
print(f"  Architecture: {deploy_config['model_name']}")
print(f"  Classes: {deploy_config['class_names']}")
print(f"  Val Acc: {deploy_config['best_val_acc']:.2%}")

if eval_path.exists():
    with open(eval_path, 'r') as f:
        eval_results = json.load(f)
    print(f"  Test Acc: {eval_results['test_accuracy']:.2%}")
    print(f"  Macro F1: {eval_results['macro_f1']:.3f}")

Deployment files verified:
  Model:   deployment/best_model.pth (42.7 MB)
  Config:  deployment/deploy_config.json
  Eval:    models/evaluation_results.json (exists: True)
  Architecture: resnet18
  Classes: ['Mild Dementia', 'Moderate Dementia', 'Non Demented', 'Very mild Dementia']
  Val Acc: 96.28%
  Test Acc: 96.82%
  Macro F1: 0.964


## 2. Test Prediction Pipeline (Dry Run)

In [4]:
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
import pandas as pd

# Simulate what app.py does
model_name = deploy_config['model_name']
num_classes = deploy_config['num_classes']

# Load model (same logic as app.py)
if model_name == 'resnet18':
    test_model = models.resnet18(weights=None)
    test_model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(512, num_classes))
elif model_name == 'efficientnet_b0':
    test_model = models.efficientnet_b0(weights=None)
    test_model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(1280, num_classes))
elif model_name == 'vgg16':
    test_model = models.vgg16(weights=None)
    test_model.classifier[6] = nn.Sequential(nn.Dropout(0.3), nn.Linear(4096, num_classes))
else:
    # BaselineCNN
    class BaselineCNN(nn.Module):
        def __init__(self, nc=4):
            super().__init__()
            self.block1 = nn.Sequential(nn.Conv2d(3,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2,2))
            self.block2 = nn.Sequential(nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2,2))
            self.block3 = nn.Sequential(nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2,2))
            self.pool = nn.AdaptiveAvgPool2d(1)
            self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(128,64), nn.ReLU(True), nn.Dropout(0.5), nn.Linear(64,nc))
        def forward(self, x):
            x = self.block1(x); x = self.block2(x); x = self.block3(x); x = self.pool(x)
            return self.classifier(x)
    test_model = BaselineCNN(num_classes)

ckpt = torch.load(DEPLOY_DIR / 'best_model.pth', map_location='cpu', weights_only=False)
test_model.load_state_dict(ckpt['model_state_dict'])
test_model.eval()

# Test prediction
test_df = pd.read_csv(Path('processed') / 'test_files.csv')
sample_path = "Data/Non Demented/OAS1_0001_MR1_mpr-1_101.jpg"

img_size = deploy_config['img_size']
transform = T.Compose([
    T.Resize((img_size, img_size)),
    T.ToTensor(),
    T.Normalize(mean=deploy_config['imagenet_mean'], std=deploy_config['imagenet_std'])
])

image = Image.open(sample_path).convert('RGB')
input_tensor = transform(image).unsqueeze(0)

with torch.no_grad():
    outputs = test_model(input_tensor)
    probs = torch.softmax(outputs, dim=1)[0]
    pred_idx = probs.argmax().item()

class_names = deploy_config['class_names']
print("Dry run prediction test:")
print(f"  Image: {sample_path}")
print(f"  Predicted: {class_names[pred_idx]}")
print(f"  Confidence: {probs[pred_idx]:.1%}")
print(f"  All probabilities:")
for i, name in enumerate(class_names):
    print(f"    {name}: {probs[i]:.1%}")
print("\nApp.py dry run PASSED!")

Dry run prediction test:
  Image: Data/Non Demented/OAS1_0001_MR1_mpr-1_101.jpg
  Predicted: Non Demented
  Confidence: 100.0%
  All probabilities:
    Mild Dementia: 0.0%
    Moderate Dementia: 0.0%
    Non Demented: 100.0%
    Very mild Dementia: 0.0%

App.py dry run PASSED!


## 3. Generate `requirements.txt`

In [5]:
requirements = """# Alzheimer's Disease Detection - Requirements
# Core ML
torch
torchvision
torchaudio

# Web App
streamlit

# Data & Visualization
numpy
pandas
matplotlib
seaborn
Pillow

# ML Utilities
scikit-learn
tqdm
"""

with open(PROJECT_DIR / 'requirements.txt', 'w') as f:
    f.write(requirements)

print("requirements.txt created.")
print("\nContents:")
print(requirements)

requirements.txt created.

Contents:
# Alzheimer's Disease Detection - Requirements
# Core ML
torch
torchvision
torchaudio

# Web App
streamlit

# Data & Visualization
numpy
pandas
matplotlib
seaborn
Pillow

# ML Utilities
scikit-learn
tqdm



## 4. Web App Architecture

```
User uploads MRI image
        |
        v
[Streamlit Frontend] (app.py)
   - Premium dark theme UI
   - Patient info fields (name, ID)
   - File uploader widget
   - Image display
   - 2 Tabs: Analyze MRI / Model Performance
        |
        v
[Preprocessing Pipeline]
   - Resize to 128x128
   - Convert to RGB
   - Normalize (ImageNet stats)
   - Convert to tensor
        |
        v
[PyTorch Model]
   - Load from deployment/best_model.pth
   - Forward pass (inference)
   - Softmax probabilities
        |
        v
[Results Display]
   - Predicted class with color-coded severity
   - Confidence score
   - Dark-themed probability bar chart
   - Detailed probability table
        |
        v
[Report Generation]
   - Self-contained HTML report
   - Patient info (name + ID)
   - Embedded MRI image (base64)
   - Prediction results & probabilities
   - Severity description
   - Disclaimer
   - Downloadable as .html file
```

## 5. App Features Breakdown

### UI Design
- **Dark theme** with gradient backgrounds (`#0f0c29` -> `#302b63` -> `#24243e`)
- **Inter font** from Google Fonts for clean, professional typography
- **Gradient text** for the hero header using CSS `background-clip`
- **Color-coded results**: Green (Non Demented), Amber (Very Mild/Mild), Red (Moderate)
- **Glassmorphism cards** with subtle borders and shadows
- **Custom probability chart** with dark background matching the theme

### Report Feature
- Click **"Download Full Report (HTML)"** after prediction
- The report is a **self-contained HTML file** (no external dependencies)
- Contains:
  - Patient name and ID
  - Analysis date/time
  - Model information
  - Prediction result with severity badge
  - Class probability bars
  - Embedded MRI scan image
  - Medical disclaimer
- Can be opened in any browser and printed to PDF

### Model Performance Tab
- Shows test set accuracy, F1, precision, recall metrics
- Per-class performance table
- Confusion matrix visualization (dark-themed)

## 6. How to Run the Web App

### Local (Your Computer)

1. **Install Streamlit** (if not already):
   ```bash
   pip install streamlit
   ```

2. **Run the app** from the project directory:
   ```bash
   streamlit run app.py
   ```

3. **Open your browser** at `http://localhost:8501`

4. **Usage:**
   - (Optional) Enter patient name and ID in the sidebar
   - Upload an MRI image in the "Analyze MRI" tab
   - View the prediction result and confidence
   - Click "Download Full Report" to get the HTML report
   - Check the "Model Performance" tab for accuracy metrics

### Cloud Deployment (Optional)

**Streamlit Cloud (Easiest):**
1. Push project to GitHub (include `app.py`, `deployment/`, `models/evaluation_results.json`, `requirements.txt`)
2. Go to [share.streamlit.io](https://share.streamlit.io)
3. Connect your GitHub repo
4. Deploy!

**Note:** The model file (~45 MB for ResNet18) may need Git LFS.

## 7. Summary

### What We Built
1. A **premium Streamlit web application** (`app.py`) with:
   - Dark gradient theme with Inter font
   - Patient information fields
   - MRI image upload and real-time AI prediction
   - Color-coded severity results (Green/Amber/Red)
   - Dark-themed probability bar chart
   - **Downloadable HTML report** with full analysis
   - Model performance tab with confusion matrix
   - Sidebar with model info and medical disclaimer
2. `requirements.txt` for easy dependency installation
3. Verified the prediction pipeline works end-to-end

### Files
- `app.py` -- Streamlit web application
- `requirements.txt` -- Python dependencies
- `deployment/best_model.pth` -- Model weights
- `deployment/deploy_config.json` -- Model configuration
- `models/evaluation_results.json` -- Test set metrics

### Final Project Structure
```
Alzeimher Project/
  Data/                       <- Raw MRI images (4 classes)
  processed/                  <- Split CSVs + config
  models/                     <- All trained model checkpoints + eval results
  deployment/                 <- Production model + config
  app.py                      <- Streamlit web app (premium UI)
  requirements.txt            <- Dependencies
  01_project_overview.ipynb   <- Dataset explanation
  02_data_exploration.ipynb   <- Visual data analysis
  03_data_preprocessing.ipynb <- Preprocessing pipeline
  04_model_training_baseline.ipynb <- Baseline CNN
  05_model_training_transfer_learning.ipynb <- Transfer learning
  06_model_evaluation.ipynb   <- Test set evaluation
  07_model_saving_and_loading.ipynb <- Inference pipeline
  08_web_app_integration.ipynb <- This notebook (web app)
```

### PROJECT COMPLETE!
You have built a complete end-to-end deep learning pipeline for Alzheimer's Disease detection from MRI brain images, with a professional web interface.

---
*End of Notebook 08 -- Project Complete*